# Run a single baseline/finetuned experiment

## Purpose of this notebook

Runs exactly one config from `experiments/baseline/` or `experiments/finetuned/` —
the same 20 configs listed in `.vscode/launch.json` for local debugging, here
launched on a Colab GPU instead. Set `EXPERIMENT` below and run the notebook;
change it and re-run to launch a different one.

Each `<Anomaly>_<Arch>` directory pairs a "baseline" config (hand-set
hyperparameters) against a "finetuned" one (hyperparameters adjusted from
that baseline — see `experiments/finetuned/`'s configs for the exact per-run
changes) for the same architecture/anomaly-type combination, so the two can
be compared directly.

## Available experiments

`EXPERIMENT` takes the form `"<tuning>/<Anomaly>_<Arch>"`:

| `<tuning>` | `<Anomaly>_<Arch>` options |
|---|---|
| `baseline` or `finetuned` | `Point_MLP`, `Point_MLP_Cyclic`, `Contextual_MLP`, `Contextual_MLP_Cyclic`, `Contextual_LSTM`, `Contextual_Transformer`, `Group_MLP`, `Group_MLP_Cyclic`, `Group_LSTM`, `Group_Transformer` |

e.g. `EXPERIMENT = "finetuned/Contextual_LSTM"` runs
`experiments/finetuned/Contextual_LSTM/era5_contextual_LSTM_config_standalone.yaml`.

### Before running

This notebook reads code and data from a **private GitHub repository**. You must:

1. Have permission to access the read-only repository with a fine-grained GitHub token.
2. In Colab, open the **Secrets** panel using the key icon.
3. Add the secret `GITHUB_TOKEN`.
4. Enable **Notebook access** for that secret.
5. Select **Runtime → Change runtime type → GPU**.
6. Set `EXPERIMENT` in the cell below, then choose **Runtime → Run all**.

The source ERA5 data and its licence information are described in `DATA_LICENSE.md`.

# Run Experiment

### Choose which experiment to run

In [ ]:
EXPERIMENT = "baseline/Contextual_LSTM"  # <-- edit this, then Runtime -> Run all

TUNING, DIRNAME = EXPERIMENT.split("/")
assert TUNING in ("baseline", "finetuned"), f"Unknown tuning {TUNING!r} (expected 'baseline' or 'finetuned')"
print(f"Selected: {EXPERIMENT}")


### Check GPU availability

In [ ]:
%matplotlib inline
import torch

print("Environment check")
print("-----------------")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Selected device:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU was detected. The notebook can still run, but "
        "training will be slower."
    )


### Import and/or load Repo

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo") if "COLAB_RELEASE_TAG" in os.environ else Path("repo")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "The Colab secret GITHUB_TOKEN is unavailable. "
            "Open the key icon in the left sidebar, add the token, "
            "and enable Notebook access."
        )

    if not REPO_DIR.exists():
        # Use an askpass helper so the token is not stored in the Git remote URL.
        askpass = Path("/content/git_askpass.sh")
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)

        env = os.environ.copy()
        env["GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"

        try:
            subprocess.run(
                ["git", "clone", REPO_URL, str(REPO_DIR)],
                check=True,
                env=env,
            )
        finally:
            askpass.unlink(missing_ok=True)

    os.chdir(REPO_DIR)
else:
    # When launched from notebooks/colab inside a local checkout,
    # move to the repository root.
    if not Path("run_regression.py").exists():
        os.chdir("..")

print("Working directory:", os.getcwd())
print("Installing the project dependencies...")
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
%env MPLBACKEND=Agg
print("Setup complete.")


### Step 1 — Locate the config and confirm the run directory

In [ ]:
import glob
import yaml

EXPERIMENT_DIR = f"experiments/{TUNING}/{DIRNAME}"
config_matches = glob.glob(f"{EXPERIMENT_DIR}/*_config_standalone.yaml")
if not config_matches:
    raise FileNotFoundError(
        f"No config found under {EXPERIMENT_DIR}/ — check EXPERIMENT is one of the "
        "<Anomaly>_<Arch> directory names listed above."
    )
CONFIG_PATH = config_matches[0]

with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)
MODEL_ARCH = _cfg["model"]["type"]

print(f"Config     : {CONFIG_PATH}")
print(f"Model arch : {MODEL_ARCH}  (cyclic_recon={_cfg['model'].get('cyclic_recon', False)})")
print(f"Run dir    : {_cfg['common']['logging']['run_dir']}")


### Step 2 — Run the experiment

Trains and streams-evaluates the selected config end to end — equivalent to
running `python -m experiments.{TUNING}.{DIRNAME}.run_era5_..._standalone` from
a terminal, or launching the matching entry in `.vscode/launch.json` locally.

##### Command:

In [ ]:
from regression.run_trial import run_trial

run_trial(CONFIG_PATH, MODEL_ARCH)


### Step 3 — Inspect the run output

In [ ]:
run_dir = _cfg["common"]["logging"]["run_dir"]
print("Run directory contents:")
for p in sorted(Path(run_dir).glob("*")):
    print(" ", p)


## Scope of this notebook

This notebook runs exactly one `experiments/{baseline,finetuned}/<Anomaly>_<Arch>/`
config per execution — set `EXPERIMENT` and re-run for another. It does not
aggregate results across configs; for a suite-style multi-run comparison with
`cross_compare.py`-style plots, see the `notebooks/cases/` notebooks instead.

Both `baseline` and `finetuned` configs read the same full-scale ERA5 split
files under `data/era5/full_scale/split/` (see `DATA_LICENSE.md`); only the
hyperparameters differ between the two tuning variants for a given
`<Anomaly>_<Arch>`.